# 01 — EDA de Features

Exploración del dataset de features para el modelo BiciMAD.

> **Nota:** El análisis completo requiere al menos 7 días de datos para que las features históricas sean representativas, y 4+ semanas para patrones temporales robustos. Con menos datos, algunas celdas mostrarán NaN o gráficos poco informativos.

In [ ]:
import sys
sys.path.insert(0, '..')  # Allow imports from project root

import polars as pl
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 5)
plt.style.use('seaborn-v0_8-whitegrid')

print(f'Polars {pl.__version__}')

## 1. Cargar dataset

In [ ]:
from pathlib import Path
import polars as pl
from src.features.build_dataset import _load_local_snapshots
from src.features.build_features import build_all_features

# Find repo root regardless of where Jupyter was launched from
_cwd = Path.cwd()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = str(REPO_ROOT / "data/raw")
print(f"Using data dir: {DATA_DIR}")

# Load raw snapshots and build features WITHOUT filtering null targets
# (build_training_dataset drops rows with null target — too aggressive for EDA)
raw_df = _load_local_snapshots(DATA_DIR, start_date=None, end_date=None)
raw_df = raw_df.filter(pl.col("activate") == 1)
df = build_all_features(raw_df)

n_null_target = df["target_dock_bikes_1h"].is_null().sum()
print(f'Filas totales (incl. target null): {len(df):,}')
print(f'  → con target: {len(df) - n_null_target:,}  |  sin target (últimas 4 por estación): {n_null_target:,}')
print(f'Columnas: {len(df.columns)}')
print(f'Estaciones: {df["station_id"].n_unique()}')
print(f'Rango temporal: {df["snapshot_timestamp"].min()} → {df["snapshot_timestamp"].max()}')

if len(df) < 10:
    print("\n⚠️  Muy pocos datos — necesitas acumular más snapshots para un EDA representativo.")
    print("   Las features históricas (avg_dock_same_hour_7d, etc.) requieren 7+ días.")

df.head()

## 2. Tipos de datos y NaN por feature

In [ ]:
from src.features.feature_definitions import ALL_FEATURES, get_features_by_group

null_counts = df.select(
    [pl.col(f.name).is_null().sum().alias(f.name) for f in ALL_FEATURES]
).transpose(include_header=True, column_names=['null_count'])

null_counts = null_counts.with_columns(
    (pl.col('null_count') / len(df) * 100).round(1).alias('null_pct')
)

print('Nulos por grupo:')
for group in ['lag', 'temporal', 'meteo', 'stats', 'static']:
    features = [f.name for f in get_features_by_group(group)]
    subset = null_counts.filter(pl.col('column').is_in(features))
    print(f'\n  [{group}]')
    print(subset.to_pandas().to_string(index=False))

## 3. Distribución del target

In [ ]:
target = df['target_dock_bikes_1h'].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2)
axes[0].hist(target, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución: dock_bikes en t+60min')
axes[0].set_xlabel('dock_bikes')
axes[0].set_ylabel('Frecuencia')

axes[1].boxplot(target, vert=True)
axes[1].set_title('Boxplot: target_dock_bikes_1h')
plt.tight_layout()
plt.show()

print(f'Media: {target.mean():.2f}  |  Mediana: {float(pl.Series(target).median()):.2f}  |  Std: {target.std():.2f}')

## 4. Patrones temporales

In [ ]:
if df['hour_of_day'].n_unique() >= 12:
    hourly = (
        df.group_by('hour_of_day')
        .agg(pl.col('dock_bikes_now').mean().alias('mean_dock_bikes'))
        .sort('hour_of_day')
    )
    plt.bar(hourly['hour_of_day'].to_numpy(), hourly['mean_dock_bikes'].to_numpy(),
            color='steelblue', edgecolor='white')
    plt.xlabel('Hora del día')
    plt.ylabel('dock_bikes_now (media)')
    plt.title('Bicicletas disponibles por hora (media de todas las estaciones)')
    plt.xticks(range(0, 24))
    plt.show()
else:
    print('Datos insuficientes para patrón horario — necesitas datos de al menos 12 horas distintas.')

## 5. Impacto del tiempo meteorológico

In [ ]:
weather_groups = df.group_by('is_raining').agg(
    pl.col('dock_bikes_now').mean().alias('mean_dock'),
    pl.col('dock_bikes_now').count().alias('n')
).sort('is_raining')

print('Impacto de la lluvia:')
print(weather_groups)

cold_groups = df.group_by('feels_cold').agg(
    pl.col('dock_bikes_now').mean().alias('mean_dock'),
    pl.col('dock_bikes_now').count().alias('n')
).sort('feels_cold')

print('\nImpacto del frío (<8°C feels-like):')
print(cold_groups)

## 6. Validación de lag features (sin leakage)

In [ ]:
lag_cols = ['dock_bikes_lag_15m', 'dock_bikes_lag_30m', 'dock_bikes_lag_1h']
lag_nulls = {col: df[col].is_null().sum() for col in lag_cols}
print('Nulos en lag features (esperado: 1 por estación x lag):')
for col, n in lag_nulls.items():
    print(f'  {col}: {n} nulos')

# Correlación lag_1h vs dock_bikes_now
valid = df.select(['dock_bikes_now', 'dock_bikes_lag_1h']).drop_nulls()
if len(valid) > 0:
    corr = valid.select(pl.corr('dock_bikes_now', 'dock_bikes_lag_1h'))[0, 0]
    print(f'\nCorrelación dock_bikes_now vs dock_bikes_lag_1h: {corr:.4f}')
    print('(Esperado: alta, > 0.90 en datos reales)')

## 7. Tasa de completitud de features históricas

In [ ]:
hist_features = ['avg_dock_same_hour_7d', 'std_dock_same_hour_7d',
                 'avg_dock_same_weekday', 'station_daily_turnover', 'dock_bikes_same_time_1w']

print('Completitud de features históricas:')
for col in hist_features:
    non_null = df[col].is_not_null().sum()
    pct = non_null / len(df) * 100
    print(f'  {col}: {non_null:,}/{len(df):,} ({pct:.1f}% non-null)')

print('\nNota: Las features históricas requieren datos de días/semanas anteriores.')
print('Con solo 1-2 días de datos, la mayoría serán NaN (comportamiento esperado).')

## 8. Correlación features vs target

> Requiere suficientes datos no-nulos en features históricas. Con pocos datos, las correlaciones históricas serán NaN.

In [ ]:
from src.features.feature_definitions import FEATURE_NAMES

numeric_features = [
    f for f in FEATURE_NAMES
    if df[f].dtype in (pl.Float64, pl.Int64, pl.Int32, pl.Float32)
]

correlations = []
for feat in numeric_features:
    valid = df.select([feat, 'target_dock_bikes_1h']).drop_nulls()
    if len(valid) > 10:
        corr = valid.select(pl.corr(feat, 'target_dock_bikes_1h'))[0, 0]
        correlations.append({'feature': feat, 'corr': corr})

if correlations:
    corr_df = pl.DataFrame(correlations).sort('corr', descending=True)
    print('Top correlaciones con target_dock_bikes_1h:')
    print(corr_df.head(15))
else:
    print('Datos insuficientes para calcular correlaciones.')

## 9. Distribución de distrito

In [ ]:
distrito_counts = (
    df.select(['station_id', 'distrito'])
    .unique()
    .group_by('distrito')
    .count()
    .sort('count', descending=True)
)
print('Estaciones por distrito:')
print(distrito_counts)
null_distritos = df.filter(pl.col('distrito').is_null())['station_id'].n_unique()
print(f'\nEstaciones sin distrito asignado: {null_distritos}')